# Phase 04: QLoRA Fine-Tuning - Code Mentor AI

In this notebook, we perform Supervised Fine-Tuning (SFT) on the **`Qwen/Qwen2.5-0.5B-Instruct`** base model using our preprocessed dataset and **QLoRA** (4-bit quantization + Low-Rank Adaptation).

### Notebook Outline:
1. Install & Import Required Libraries
2. Environment & Device Configuration
3. Load Preprocessed Dataset
4. Format Chat Messages for SFT
5. Load Tokenizer & Quantized Base Model
6. Configure PEFT (LoRA)
7. Set Up Training Arguments & SFTTrainer
8. Execute Fine-Tuning
9. Save Fine-Tuned LoRA Adapter

# Getting dataset location

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/dataset_dict.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/validation/state.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/validation/dataset_info.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/validation/data-00000-of-00001.arrow
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/test/state.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/test/dataset_info.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/test/data-00000-of-00001.arrow
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/train/state.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/train/dataset_info.json
/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data/tra

# 1. Install Required Libraries

In [2]:
!pip install -q trl peft bitsandbytes accelerate transformers datasets
print('Libraries installed successfully!')

Libraries installed successfully!


# 2. Imports and Device Setup 

In [3]:
# 2. Imports and Device Setup (Must set environment variable BEFORE importing torch)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk

print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training device: {device.upper()}")
if device == "cuda":
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
Training device: CUDA
GPU Device: Tesla T4


# 3. Load Preprocessed Dataset from Disk

In [4]:
dataset_path = "/kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data"
print(f"Loading dataset from: {dataset_path}")

dataset = load_from_disk(dataset_path)
print("Dataset loaded successfully!")
print(dataset)

Loading dataset from: /kaggle/input/datasets/tanishtushid/code-mentor-ai/processed_code_mentor_data
Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 100064
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5559
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 5560
    })
})


# 4. Format Chat Messages into Training Text 

In [5]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def formatting_prompts_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Make sure the working cache dir exists
os.makedirs("/kaggle/working/cache", exist_ok=True)

print("Formatting dataset splits with chat template...")

encoded_dataset = {}
for split in dataset:                      # 'train', 'validation', 'test'
    print(f" processing split: {split}")
    encoded_dataset[split] = dataset[split].map(
        formatting_prompts_func,
        cache_file_name=f"/kaggle/working/cache/{split}_formatted.arrow",  # writable path
        load_from_cache_file=False,          # avoid stale cache if you re-run
        desc=f"Formatting {split}",
    )

print("Done")
print("\nSample formatted text (train[0]):")
print(encoded_dataset['train'][0]['text'][:500])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Formatting dataset splits with chat template...
 processing split: train


... (2 outputs left out)

Formatting validation:   0%|          | 0/5559 [00:00<?, ? examples/s]

 processing split: test


Formatting test:   0%|          | 0/5560 [00:00<?, ? examples/s]

Done

Sample formatted text (train[0]):
<|im_start|>system
You are Code Mentor AI, an expert software engineer and programming tutor. Your goal is to help users write clean code, explain complex algorithms step-by-step, and debug errors thoroughly.<|im_end|>
<|im_start|>user
change the following snippet to php:
def odd_count_elements(list1, list2):
    counts = {}
    for i in list1 + list2:
        if i in counts:
            counts[i] += 1
        else:
            counts[i] = 1

    odds = [i for i in counts if counts[i] % 2 != 0]



# 5. Load Tokenizer & Quantized Base Model (QLoRA 4-bit)

In [6]:
print(f"Configuring 4-bit quantization for {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Explicitly set torch_dtype=torch.float16 so Qwen doesn't default to bfloat16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)
print("Base model loaded in float16 and prepared for QLoRA successfully!")

Configuring 4-bit quantization for Qwen/Qwen2.5-0.5B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded in float16 and prepared for QLoRA successfully!


# 6. Configure PERT (LoRA Settings)

In [7]:
peft_config = LoraConfig(
    r=16,                              # Rank dimension
    lora_alpha=32,                     # Scaling parameter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,                 # Dropout probability to prevent overfitting
    bias="none",
    task_type="CAUSAL_LM"
)

print("PEFT LoRA configuration created successfully!")

PEFT LoRA configuration created successfully!


# 7. Set Up Training Arguments & SFTTrainer

In [8]:
# SFT = Supervised Fine-Tuning, used to train an LLM on instruction-response examples.

from trl import SFTConfig, SFTTrainer

# Directory where model checkpoints and training outputs will be stored.
output_dir = "/kaggle/working/code_mentor_checkpoints"

# Configure all training hyperparameters and settings.
training_args = SFTConfig(
    output_dir=output_dir,  # Save checkpoints, logs, and training outputs in this directory.

    per_device_train_batch_size=2,  # Process 2 training examples per GPU in each microbatch.
    per_device_eval_batch_size=2,   # Process 2 validation examples per GPU during evaluation.

    gradient_accumulation_steps=4,  # Accumulate gradients across 4 microbatches before an optimizer update.
                                    # Effective batch size = 2 × 4 = 8 examples on one GPU.

    learning_rate=2e-4,  # Learning rate = 0.0002; controls the size of optimizer updates.
                         # This is a common starting value for LoRA adapter training.

    logging_steps=50,  # Log training metrics every 50 training steps.

    num_train_epochs=1,  # Train for 1 complete pass through the dataset, unless max_steps stops or extends training.

    max_steps=1500,  # Perform at most 1500 optimizer update steps.
                     # Because this is positive, it takes precedence over num_train_epochs.

    optim="paged_adamw_8bit",  # Use an 8-bit AdamW optimizer with paged memory management.
                              # Helps reduce optimizer-state memory usage.

    fp16=False,  # Disable FP16 mixed-precision training.
                 # Training uses the default model/compute dtype instead of FP16 AMP.

    bf16=False,  # Disable BF16 mixed-precision training.
                 # The Trainer will not enable BF16 AMP.

    eval_strategy="steps",  # Evaluate the model after a fixed number of training steps.

    eval_steps=200,  # Run validation every 200 training steps to monitor generalization.

    save_strategy="steps",  # Save model checkpoints at regular training-step intervals.

    save_steps=200,  # Save a checkpoint every 200 training steps.

    save_total_limit=2,  # Retain at most 2 checkpoints; older checkpoints may be deleted.

    load_best_model_at_end=True,  # Reload the best checkpoint after training.
                                  # With default settings, eval loss is typically the selection metric.

    report_to="none",  # Disable integrations such as Weights & Biases for experiment tracking.

    dataset_text_field="text",  # Read training text from the dataset column named "text".
                                # Each example should contain the correctly formatted text for SFT.

    max_length=1024,  # Limit each training sequence to 1024 tokens.
                     # Longer sequences may be truncated, depending on dataset preparation and TRL version.
)

# Manually override Trainer's internal GPU-count attribute.
# Intended to avoid the nn.DataParallel error when multiple GPUs are visible.
training_args._n_gpu = 1

# Create the supervised fine-tuning trainer.
trainer = SFTTrainer(
    model=model,  # The pretrained base language model, ideally loaded in 4-bit for QLoRA.

    train_dataset=encoded_dataset["train"],  # Dataset split used to update the model's trainable parameters.

    eval_dataset=encoded_dataset["validation"],  # Held-out dataset split used to measure validation performance.

    peft_config=peft_config,  # LoRA configuration defining adapter rank, target modules, and related settings.
                              # SFTTrainer applies these adapters to the base model.

    processing_class=tokenizer,  # Tokenizer used to prepare input text and construct model inputs.

    args=training_args,  # Pass the complete training configuration to SFTTrainer.
)

# Print a confirmation message after successful trainer initialization.
print("SFTTrainer initialized successfully with T4-safe precision settings!")

Tokenizing train dataset:   0%|          | 0/100064 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100064 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100064 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100064 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5559 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/5559 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5559 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/5559 [00:00<?, ? examples/s]

SFTTrainer initialized successfully with T4-safe precision settings!


# 8. Execute Fine-Tuning

In [9]:
print("Starting QLoRA fine-tuning for Code Mentor AI...")
trainer.train()
print("Fine-tuning completed successfully!")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting QLoRA fine-tuning for Code Mentor AI...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,1.323192,1.261575,1.255717,0.692061,889770.000000
400,1.252694,1.250277,1.252877,0.694196,1754372.000000
600,1.252540,1.239574,1.246266,0.696110,2632990.000000
800,1.205531,1.232738,1.233220,0.697707,3512819.000000
1000,1.220100,1.227749,1.231110,0.698866,4380412.000000
1200,1.228816,1.223832,1.231786,0.699327,5253198.000000
1400,1.185894,1.222041,1.228634,0.699744,6124991.000000
1500,1.208101,1.221932,1.228741,0.699775,6557169.000000


Fine-tuning completed successfully!


# 9. Save the Fine-Tuned LoRA Adapter


In [10]:
final_model_path = "/kaggle/working/code_mentor_lora_model"

print(f"Saving fine-tuned LoRA adapter to {final_model_path}...")
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("LoRA adapter and tokenizer saved successfully!")

Saving fine-tuned LoRA adapter to /kaggle/working/code_mentor_lora_model...
LoRA adapter and tokenizer saved successfully!
